# Module 05 Lab — Multi-Step Research Workflow

**Course:** AI Agents Teaching Platform  
**Module:** 05 — Workflow Patterns & Control Flow  
**Estimated time:** 2–3 hours

---

### What you'll build

A three-stage research pipeline:
1. **Planner** — turns a question into 3–5 structured search queries
2. **Searcher** — fans out in parallel with `asyncio.gather`
3. **Synthesiser** — merges results into a cited draft
4. **Verifier** — checks citations, triggers up to 2 repair passes

### Hard constraints
- Max 2 repair iterations — enforced in code
- Planner must return a Python list, not free text
- Searcher must use `asyncio.gather` — not a sequential loop
- Runner prints: stages, iteration count, estimated token cost

## Step 1 — Choose your model

| Provider | Model string | Key env var |
|---|---|---|
| Anthropic | `claude-haiku-4-5-20251001` | `ANTHROPIC_API_KEY` |
| OpenAI | `gpt-4o-mini` | `OPENAI_API_KEY` |
| Google Gemini | `gemini/gemini-1.5-flash` | `GEMINI_API_KEY` |

In [ ]:
MODEL = "claude-haiku-4-5-20251001"
print(f"Model: {MODEL}")

In [ ]:
%pip install -q litellm

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your API key: ")
print("Key set ✓")

## Setup: shared LiteLLM call + token tracking

All four stages call the same `_call()` helper. It also accumulates token usage
so the runner can print an estimated cost at the end.

In [ ]:
import json
import asyncio
import litellm

litellm.drop_params = True

USAGE = {"input": 0, "output": 0}

def _call(messages: list, max_tokens: int = 512) -> str:
    response = litellm.completion(model=MODEL, messages=messages, max_tokens=max_tokens)
    USAGE["input"]  += response.usage.prompt_tokens
    USAGE["output"] += response.usage.completion_tokens
    return response.choices[0].message.content

print("Shared call helper defined ✓")

## Step 4 — Planner

The planner is complete — run it to verify it returns a list (not a string).

In [ ]:
def plan_queries(question: str) -> list:
    raw = _call([{"role": "user", "content": (
        f"Turn this research question into 3-5 targeted search queries.\n"
        f'Return JSON only: {{"queries": ["query 1", ...]}}\n\nQuestion: {question}'
    )}])
    try:
        data = json.loads(raw)
        return data["queries"]
    except (json.JSONDecodeError, KeyError):
        # Fallback: treat each line as a query
        return [line.strip("- ") for line in raw.strip().splitlines() if line.strip()]

# Quick test
test_queries = plan_queries("What are the main causes of context window overflow in LLM applications?")
print(f"Planner returned {len(test_queries)} queries:")
for q in test_queries:
    print(f"  - {q}")
assert isinstance(test_queries, list), "Planner must return a list"
print("Planner test passed ✓")

## Step 5 — Parallel searcher (TODO)

The `search_one` function is complete. Fill in `search_all` to:
1. Use `asyncio.gather` with `return_exceptions=True` to run all queries in parallel
2. Handle individual branch failures gracefully — a failed branch returns `{"query": q, "facts": [], "source": "failed"}`
3. Return a list of result dicts, one per query

In [ ]:
async def search_one(query: str) -> dict:
    raw = _call([{"role": "user", "content": (
        f'Simulate a search result for this query. Return 2-3 key facts as JSON.\n'
        f'{{"query": "...", "facts": ["fact 1", ...], "source": "simulated"}}\n\nQuery: {query}'
    )}])
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"query": query, "facts": [raw], "source": "simulated"}


async def search_all(queries: list) -> list:
    """TODO: use asyncio.gather to run search_one for all queries in parallel.
    Use return_exceptions=True so one failed branch doesn't crash everything.
    Replace failed branches with {"query": q, "facts": [], "source": "failed"}.
    """
    # TODO
    pass


# Quick test
test_results = asyncio.run(search_all(test_queries[:2]))
print(f"Searcher returned {len(test_results)} results")
assert len(test_results) == 2, "Must return one result per query"
print("Searcher test passed ✓")

<details>
<summary>💡 Stuck? Reveal the search_all solution</summary>

```python
async def search_all(queries: list) -> list:
    results = await asyncio.gather(
        *[search_one(q) for q in queries],
        return_exceptions=True,
    )
    ok = []
    for query, result in zip(queries, results):
        if isinstance(result, Exception):
            print(f"  branch failed for '{query}': {result}")
            ok.append({"query": query, "facts": [], "source": "failed"})
        else:
            ok.append(result)
    return ok
```

`return_exceptions=True` is the whole trick: failures come back as values instead of propagating.

</details>

## Step 6 — Synthesiser and verifier

Both are complete — read and understand them before running the full pipeline.

In [ ]:
def synthesise(question: str, search_results: list) -> str:
    results_text = json.dumps(search_results, indent=2)
    return _call([{"role": "user", "content": (
        f"Write a 300-word answer to the question using only the provided search results.\n"
        f"Rules:\n- Every factual claim must include a citation: [Source: <source>]\n"
        f"- Do not invent facts not in the results\n- End with a 'Sources used:' list\n"
        f"\nQuestion: {question}\n\nSearch results:\n{results_text}"
    )}], max_tokens=800)


MAX_ITERATIONS = 2

def verify(draft: str) -> dict:
    raw = _call([{"role": "user", "content": (
        f'Check this draft for missing citations.\n'
        f'Return JSON: {{"pass": true/false, "missing_citations": ["sentence...", ...]}}\n\nDraft:\n{draft}'
    )}])
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"pass": True, "missing_citations": []}  # fail-open to respect MAX_ITERATIONS


def repair(draft: str, missing: list) -> str:
    missing_text = "\n".join(f"- {s}" for s in missing)
    return _call([{"role": "user", "content": (
        f"Add citations to these sentences. Return the full revised draft.\n"
        f"Sentences needing citations:\n{missing_text}\n\nDraft:\n{draft}"
    )}], max_tokens=800)


def verify_with_repair(draft: str) -> tuple:
    iteration = 0
    while iteration < MAX_ITERATIONS:
        result = verify(draft)
        if result["pass"]:
            break
        draft = repair(draft, result["missing_citations"])
        iteration += 1
    return draft, iteration


print("Synthesiser and verifier defined ✓")

## Step 7 — Wire the full pipeline

In [ ]:
def run(question: str):
    USAGE["input"] = USAGE["output"] = 0
    print(f"\n{'='*60}\nResearch: {question}\n{'='*60}")

    print("[1/4] Planning queries...")
    queries = plan_queries(question)
    print(f"  {len(queries)} queries generated")

    print("[2/4] Searching (parallel)...")
    results = asyncio.run(search_all(queries))
    print(f"  {len(results)} result sets received")

    print("[3/4] Synthesising...")
    draft = synthesise(question, results)

    print("[4/4] Verifying citations...")
    final, iterations = verify_with_repair(draft)
    print(f"  {iterations} repair iteration(s)")

    # Cost estimate (blended price — adjust to your provider)
    est_cost = USAGE["input"] / 1e6 * 3 + USAGE["output"] / 1e6 * 15
    print(f"\nStages: planning → parallel search ({len(queries)} branches) → synthesis → verification ({iterations} iter)")
    print(f"Tokens: {USAGE['input']} in / {USAGE['output']} out (~${est_cost:.4f})")
    print("\n--- Final answer ---")
    print(final)
    return final


answer = run("What are the main causes of context window overflow in LLM applications?")

## Step 8 — Experiments

### Experiment A: Verifier false pass
In `synthesise`, add `"Do not include any citations."` to the prompt. Run the pipeline.
Does the verifier catch all missing citations, or does it false-pass?

### Experiment B: Searcher failure
Add `raise ValueError("simulated failure")` inside `search_one` for the first query.
Verify the pipeline still completes — the failed branch should produce `{"source": "failed"}`.

### Experiment C: Iteration cap
Change `MAX_ITERATIONS = 2` to `MAX_ITERATIONS = 0`. What does the pipeline return?

### Experiment D: Swap the model
Change `MODEL` to `gpt-4o-mini`. Compare output quality and token cost.

In [ ]:
# Scratch cell — use for experiments


## Step 9 — Reflection

1. Draw the control flow diagram for this pipeline (include the repair loop and its stopping conditions).
2. Why does `verify()` fail-open (`{"pass": True}`) on a JSON parse error, rather than raising?
3. The verifier checks citation *presence*, not citation *grounding*. What does this mean? How would you fix it?
4. What changes if one of the 5 searcher branches returns `{"facts": [], "source": "failed"}`?

*(Double-click to edit)*

1. 
2. 
3. 
4. 